## Modelo de machine learning basico

**Punto clave:** Dado que la variable objetivo (`Sales`) es un valor numérico continuo, estamos frente a un problema de **Regresión**, no de Clasificación. Por lo tanto, los algoritmos utilizados cambian (ej. de `RandomForestClassifier` a `RandomForestRegressor`) y la métrica de evaluación pasa a ser el error (ej. MSE) o el $R^2$.

Además, como este dataset solo contiene variables numéricas (`TV`, `Radio`, `Newspaper`), he incluido un paso donde **creamos artificialmente una variable categórica** (discretización) para que puedas demostrar las técnicas de codificación.

### 1. Carga de Datos y Preprocesamiento (Creación de Categorías)

Este bloque demuestra cómo cargar el archivo y cómo aplicar **Feature Engineering** para transformar una variable continua (inversión en TV) en una variable categórica (Baja, Media, Alta), lo cual es útil para segmentación comercial.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import KBinsDiscretizer, OneHotEncoder

In [ ]:
# 0. Configuración inicial con los datos de publicidad
url:str = 'https://gist.githubusercontent.com/Andru-1987/42a69933ca3247f5929bbe72d2467d86/raw/108783fb4e8e624cd9f45e3ae72d81e5b023bb0c/advertising.csv' 

# 1. Cargar el dataset
df = pd.read_csv(url, index_col=0)

# Separar features (X) y target (y)
X = df[['TV', 'Radio', 'Newspaper']]
y = df['Sales']

print("Vista inicial del dataset:")
X.head()


In [ ]:

# 2. Discretización (Binning): Convertir 'TV' en categórica para demostración
# Divide la inversión de TV en 3 grupos: Baja, Media, Alta
kbins = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='quantile')
df['TV_Categoria'] = kbins.fit_transform(df[['TV']])

print("Dataset con nueva característica categórica (0=Baja, 1=Media, 2=Alta):")
df[['TV', 'TV_Categoria']].head()


In [ ]:

# 3. One-Hot Encoding sobre la nueva categoría
ohe = OneHotEncoder(sparse_output=False, drop='first')
tv_ohe = ohe.fit_transform(df[['TV_Categoria']])

# Convertir a DataFrame para mostrar a los alumnos
ohe_df = pd.DataFrame(tv_ohe, columns=['TV_Media', 'TV_Alta'])
print("Transformación One-Hot Encoding (evitando multicolinealidad):")
ohe_df.head()

### 2. Ingeniería de Características (Interacciones)

En publicidad, la suma de TV y Radio suele tener un efecto sinérgico (el impacto combinado es mayor que la suma individual). Este bloque muestra cómo Scikit-Learn detecta estas interacciones matemáticamente.


## Que hace todo esto?

Este fragmento crea **variables de interacción** a partir de las tres columnas originales del dataset (`TV`, `Radio`, `Newspaper`). Una interacción es simplemente el producto entre dos variables — captura el efecto conjunto de invertir en dos canales a la vez, algo que un modelo lineal no puede "ver" si solo le das las variables por separado. ## Línea por línea

**`poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)`**
- `degree=2` le dice que multiplique variables de a pares (no de a tres).
- `interaction_only=True` es la clave: sin esto, `PolynomialFeatures` también generaría `TV²`, `Radio²`, `Newspaper²` (potencias de cada variable consigo misma). Con `interaction_only=True` esos cuadrados se descartan y solo quedan los productos cruzados entre variables distintas.
- `include_bias=False` evita que agregue una columna extra de puros unos (el término constante de un polinomio), que en scikit-learn ya no se necesita porque el modelo maneja el intercepto por su cuenta.

**`X_interacciones = poly.fit_transform(X)`**
Ajusta la transformación y la aplica en un solo paso: toma tus 3 columnas y devuelve una matriz numpy con 6 columnas (las 3 originales más los 3 productos cruzados posibles: TV×Radio, TV×Newspaper, Radio×Newspaper).

**`poly.get_feature_names_out(...)`**
Solo sirve para ponerle nombre a esas 6 columnas nuevas, porque `fit_transform` devuelve un array de numpy sin encabezados — sin este método no sabrías cuál columna es cuál.

## Por qué se hace esto (la idea de "sinergia")

En marketing, invertir en TV y en Radio al mismo tiempo puede generar más ventas que la simple suma de invertir en cada uno por separado — por ejemplo, alguien ve el comercial de TV y luego lo refuerza escuchándolo en la radio. Un modelo lineal simple (`Sales = a·TV + b·Radio + c·Newspaper`) no puede capturar ese efecto combinado: solo suma contribuciones independientes.

Al agregar la columna `TV*Radio`, le das al modelo una nueva variable que crece justamente cuando *ambas* inversiones son altas al mismo tiempo. Si el modelo le asigna un coeficiente positivo a esa columna, está diciendo "cuando TV y Radio suben juntos, las ventas suben más de lo que explicarían por separado" — eso es matemáticamente cómo se representa la sinergia.


In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# Generar interacciones polinómicas (grado 2)
# interaction_only=True evita generar TV^2, Radio^2, solo genera TV*Radio

poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_interacciones = poly.fit_transform(X)

print("\nNuevas variables (Features) generadas:")
print(poly.get_feature_names_out(['TV', 'Radio', 'Newspaper']))

print("\nMatriz con la interacción sinérgica calculada (Primeras 2 filas):")
X_interacciones[:2]

### Regresión lineal base (baseline)

Con la matriz `X_interacciones`, entrenamos un modelo lineal simple sobre esas variables (las 3 originales más los 3 productos cruzados). Además, comparamos contra un modelo lineal entrenado solo con las variables originales, para ver cuánto aporta agregar las interacciones.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


In [ ]:

# 1. Split de train/test (mismo random_state para comparar ambos modelos sobre las mismas filas)
X_train, X_test, y_train, y_test = train_test_split(
    X_interacciones, y, test_size=0.2, random_state=42
)

X_train_orig, X_test_orig, _, _ = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2. Modelo base CON interacciones
modelo_base = LinearRegression()
modelo_base.fit(X_train, y_train)
y_pred = modelo_base.predict(X_test)

# 3. Modelo simple SIN interacciones, para comparar
modelo_simple = LinearRegression()
modelo_simple.fit(X_train_orig, y_train)
y_pred_simple = modelo_simple.predict(X_test_orig)

print("--- Comparacion de modelos en test ---")
print(f"Sin interacciones  -> R2: {r2_score(y_test, y_pred_simple):.4f} | MSE: {mean_squared_error(y_test, y_pred_simple):.4f}")
print(f"Con interacciones  -> R2: {r2_score(y_test, y_pred):.4f} | MSE: {mean_squared_error(y_test, y_pred):.4f}")


In [ ]:

# 4. Interpretar los coeficientes del modelo con interacciones
nombres_features = poly.get_feature_names_out(['TV', 'Radio', 'Newspaper'])
coeficientes = pd.DataFrame({
    'Feature': nombres_features,
    'Coeficiente': modelo_base.coef_
}).sort_values(by='Coeficiente', ascending=False)

print("\nCoeficientes del modelo (impacto estimado sobre Sales):")
print(coeficientes)
print(f"\nIntercepto (Sales base sin inversion): {modelo_base.intercept_:.4f}")

### 3. Optimización de Hiperparámetros (Búsqueda de la Mejor Regresión)

Aquí demuestras cómo encontrar la mejor configuración para un modelo predictivo de ventas utilizando **GridSearchCV**. Al ser regresión, usamos `neg_mean_squared_error` (buscamos el error más cercano a cero) o `r2` (buscamos acercarnos a 1).

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression


In [ ]:

# Iniciar el modelo base
linear_model = LinearRegression()

# Definir el espacio de búsqueda (Hiperparámetros a probar)
# Los hiperparámetros válidos son: copy_X, fit_intercept, n_jobs, positive, tol
param_grid = {
    'fit_intercept': [True, False],  # Probar con y sin intercepto
    'positive': [True, False]        # Forzar coeficientes positivos o no
}

# Scoring múltiple: pedimos r2 y neg_mean_squared_error a la vez
# (sklearn siempre maximiza, por eso el MSE se pide como "negativo")
scoring = {
    'r2': 'r2',
    'neg_mse': 'neg_mean_squared_error'
}

# Configurar GridSearchCV

grid_search = GridSearchCV(
    estimator=linear_model,
    param_grid=param_grid,
    cv=5, # Validación cruzada de 5 pliegues
    scoring=scoring, 
    refit='r2',  # con que metrica se re-entrena el mejor modelo al final
    n_jobs=-1
)



# Entrenar evaluando todas las combinaciones
grid_search.fit(X, y)

# Índice del mejor candidato dentro de cv_results_
best_idx = grid_search.best_index_
resultados = grid_search.cv_results_

r2_mean = resultados['mean_test_r2'][best_idx]
r2_std = resultados['std_test_r2'][best_idx]
mse_mean = -resultados['mean_test_neg_mse'][best_idx]  # se revierte el signo
mse_std = resultados['std_test_neg_mse'][best_idx]

print("\n--- Resultados de la Optimización ---")
print("Mejores Hiperparámetros encontrados:")
print(grid_search.best_params_)
print(f"\nR2 promedio en los 5 folds: {r2_mean:.4f} (+/- {r2_std:.4f})")
print(f"MSE promedio en los 5 folds: {mse_mean:.4f} (+/- {mse_std:.4f})")



In [ ]:
grid_search_results = pd.DataFrame(grid_search.cv_results_)
grid_search_results

In [ ]:
# Entrenar evaluando todas las combinaciones
grid_search.fit(X_train_orig, y_train)

# Índice del mejor candidato dentro de cv_results_
best_idx = grid_search.best_index_
resultados = grid_search.cv_results_

r2_mean = resultados['mean_test_r2'][best_idx]
r2_std = resultados['std_test_r2'][best_idx]
mse_mean = -resultados['mean_test_neg_mse'][best_idx]  # se revierte el signo
mse_std = resultados['std_test_neg_mse'][best_idx]

print("\n--- Resultados de la Optimización ---")
print("Mejores Hiperparámetros encontrados:")
print(grid_search.best_params_)
print(f"\nR2 promedio en los 5 folds: {r2_mean:.4f} (+/- {r2_std:.4f})")
print(f"MSE promedio en los 5 folds: {mse_mean:.4f} (+/- {mse_std:.4f})")



### 4. Arquitectura de Producción: Pipeline Completo

Integra todo el flujo. Estructura el código para producción, encapsulando la creación de interacciones, el escalado de variables numéricas y el modelo predictivo en un solo objeto para evitar el _Data Leakage_ (fuga de datos).

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint


In [ ]:

# 1. Construir el Pipeline
# El flujo entra puro y se transforma secuencialmente
pipeline_ventas = Pipeline([
    ('interacciones', PolynomialFeatures(degree=2, include_bias=False)),
    ('escalador', StandardScaler()),
    ('modelo', RandomForestRegressor(random_state=42))
])

# 2. Espacio de búsqueda apuntando a los pasos del pipeline
# Nota para la clase: Se usa el prefijo 'modelo__' para indicar a qué paso pertenece el parámetro
param_dist_pipe = {
    'modelo__n_estimators': randint(100, 300),
    'modelo__max_depth': randint(5, 20)
}

scoring = {
    'r2': 'r2',
    'neg_mse': 'neg_mean_squared_error'
}

# 3. Optimización eficiente con RandomizedSearchCV sobre el Pipeline
random_pipe = RandomizedSearchCV(
    estimator=pipeline_ventas,
    param_distributions=param_dist_pipe,
    n_iter=10, # Probar 10 combinaciones aleatorias
    cv=5,

    scoring=scoring,
    refit='r2',
    n_jobs=-1,
    random_state=42
)

# 4. Entrenar todo el flujo
random_pipe.fit(X, y)

print("\n--- Resultados del Pipeline de Producción ---")
print("Mejor configuración del pipeline:")
print(random_pipe.best_params_)
print(f"Mejor Score R2 en validación cruzada: {random_pipe.best_score_:.4f}")

# Simulación de predicción con un dato nuevo:
nuevo_presupuesto = pd.DataFrame({'TV': [150], 'Radio': [40], 'Newspaper': [20]})
prediccion = random_pipe.predict(nuevo_presupuesto)
print(f"\nPredicción de ventas para el nuevo presupuesto: {prediccion[0]:.2f} unidades")

In [ ]:
# 4b. Entrenar todo el flujo
random_pipe.fit(X_train_orig, y_train)

print("\n--- Resultados del Pipeline de Producción ---")
print("Mejor configuración del pipeline:")
print(random_pipe.best_params_)
print(f"Mejor Score R2 en validación cruzada: {random_pipe.best_score_:.4f}")

# Simulación de predicción con un dato nuevo:
nuevo_presupuesto = pd.DataFrame({'TV': [150], 'Radio': [40], 'Newspaper': [20]})
prediccion = random_pipe.predict(nuevo_presupuesto)
print(f"\nPredicción de ventas para el nuevo presupuesto: {prediccion[0]:.2f} unidades")

---
### Comparación de modelos avanzados (Random Forest, Gradient Boosting, SVR)

Hasta ahora comparamos regresión lineal con y sin interacciones. Ahora sumamos modelos no lineales para ver si mejoran el ajuste: un Random Forest sin tunear (baseline), un Random Forest optimizado, un Gradient Boosting optimizado y un SVR optimizado. Usamos Plotly para inspeccionar interactivamente los hiperparámetros de cada modelo al pasar el mouse, y para comparar visualmente qué tan cerca quedan las predicciones del valor real.

Reutilizamos `X_train_orig`, `X_test_orig`, `y_train`, `y_test` ya definidos antes (mismo split, mismas variables originales sin interacciones).

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_squared_error
import plotly.express as px
import plotly.graph_objects as go


In [ ]:

# Lista para almacenar los resultados y graficarlos luego
resultados_modelos = []
predicciones = {'Real': y_test.values}

# ==========================================
# MODELO 1: Random Forest (Base - Sin tunear)
# ==========================================
lr_base = LinearRegression()
lr_base.fit(X_train_orig, y_train)
y_pred_base = lr_base.predict(X_test_orig)

resultados_modelos.append({
    'Modelo': '1. Linear Regression (Base)',
    'R2_Score': r2_score(y_test, y_pred_base),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_base)),
    'Hiperparametros': 'Valores por defecto de Scikit-Learn'
})

predicciones['LR_Base'] = y_pred_base

# ==========================================
# MODELO 2: Random Forest (Optimizado)
# ==========================================
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5]
}
grid_rf = GridSearchCV(RandomForestRegressor(random_state=42), param_grid_rf, cv=5, scoring='r2', n_jobs=-1)
grid_rf.fit(X_train_orig, y_train)
y_pred_rf_opt = grid_rf.predict(X_test_orig)

resultados_modelos.append({
    'Modelo': '2. Random Forest (Optimizado)',
    'R2_Score': r2_score(y_test, y_pred_rf_opt),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_rf_opt)),
    'Hiperparametros': str(grid_rf.best_params_)
})
predicciones['RF_Opt'] = y_pred_rf_opt

# ==========================================
# MODELO 3: Gradient Boosting (Optimizado)
# ==========================================
param_grid_gb = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5]
}
grid_gb = GridSearchCV(GradientBoostingRegressor(random_state=42), param_grid_gb, cv=5, scoring='r2', n_jobs=-1)
grid_gb.fit(X_train_orig, y_train)
y_pred_gb_opt = grid_gb.predict(X_test_orig)

resultados_modelos.append({
    'Modelo': '3. Gradient Boosting (Opt)',
    'R2_Score': r2_score(y_test, y_pred_gb_opt),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_gb_opt)),
    'Hiperparametros': str(grid_gb.best_params_)
})
predicciones['GB_Opt'] = y_pred_gb_opt

# ==========================================
# MODELO 4: Support Vector Regression (SVR - Optimizado)
# ==========================================
param_grid_svr = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}
grid_svr = GridSearchCV(SVR(), param_grid_svr, cv=5, scoring='r2', n_jobs=-1)
grid_svr.fit(X_train_orig, y_train)
y_pred_svr_opt = grid_svr.predict(X_test_orig)

resultados_modelos.append({
    'Modelo': '4. SVR (Optimizado)',
    'R2_Score': r2_score(y_test, y_pred_svr_opt),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_svr_opt)),
    'Hiperparametros': str(grid_svr.best_params_)
})
predicciones['SVR_Opt'] = y_pred_svr_opt

df_resultados = pd.DataFrame(resultados_modelos)
df_resultados

## Evitar duplicacion de codigo

In [ ]:

def evaluar_modelo(nombre, estimador, param_grid=None, cv=5):
    """Entrena (con GridSearchCV si hay param_grid, sino directo) y devuelve metricas + predicciones."""
    if param_grid:
        buscador = GridSearchCV(estimador, param_grid, cv=cv, scoring='r2', n_jobs=-1)
        buscador.fit(X_train_orig, y_train)
        modelo_final = buscador.best_estimator_
        hiperparametros = str(buscador.best_params_)
    else:
        estimador.fit(X_train_orig, y_train)
        modelo_final = estimador
        hiperparametros = 'Valores por defecto de Scikit-Learn'

    y_pred = modelo_final.predict(X_test_orig)
    metricas = {
        'Modelo': nombre,
        'R2_Score': r2_score(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'Hiperparametros': hiperparametros
    }
    return metricas, y_pred


# Cada tupla es: (nombre a mostrar, clave para el dict de predicciones, estimador, param_grid o None)
configuraciones = [
    ('1. Linear Regression (Base)', 'LR_Base', LinearRegression(), None),
    ('2. Random Forest (Optimizado)', 'RF_Opt', RandomForestRegressor(random_state=42), {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 5, 10],
        'min_samples_split': [2, 5]
    }),
    ('3. Gradient Boosting (Opt)', 'GB_Opt', GradientBoostingRegressor(random_state=42), {
        'n_estimators': [100, 200],
        'learning_rate': [0.01, 0.1, 0.2],
        'max_depth': [3, 5]
    }),
    ('4. SVR (Optimizado)', 'SVR_Opt', SVR(), {
        'C': [0.1, 1, 10, 100],
        'kernel': ['linear', 'rbf'],
        'gamma': ['scale', 'auto']
    }),
]

resultados_modelos = []
predicciones = {'Real': y_test.values}

for nombre, clave, estimador, param_grid in configuraciones:
    metricas, y_pred = evaluar_modelo(nombre, estimador, param_grid)
    resultados_modelos.append(metricas)
    predicciones[clave] = y_pred

df_resultados = pd.DataFrame(resultados_modelos)
df_resultados


In [ ]:
# Grafico de barras interactivo: R2 de cada modelo, con hiperparametros en el hover
fig_r2 = px.bar(
    df_resultados,
    x='Modelo',
    y='R2_Score',
    color='Modelo',
    text='R2_Score',
    hover_data={'Hiperparametros': True, 'Modelo': False},
    title="Comparacion de Rendimiento Predictivo (R2 Score) - Mas alto es mejor",
    labels={'R2_Score': 'Coeficiente de Determinacion (R2)'}
)
fig_r2.update_traces(texttemplate='%{text:.4f}', textposition='outside')
fig_r2.update_layout(yaxis_range=[0, 1.1])
fig_r2.show()

In [ ]:
# Grafico de dispersion: ventas reales vs predichas (modelo base vs modelo optimizado)
fig_scatter = go.Figure()

# Linea ideal (prediccion perfecta, y = x)
rango_min = min(y_test) - 2
rango_max = max(y_test) + 2
fig_scatter.add_trace(go.Scatter(
    x=[rango_min, rango_max], y=[rango_min, rango_max],
    mode='lines',
    name='Prediccion Perfecta',
    line=dict(color='black', dash='dash')
))

# Puntos del modelo base
fig_scatter.add_trace(go.Scatter(
    x=y_test, y=y_pred_base,
    mode='markers',
    name='LR Base',
    marker=dict(color='red', size=8, opacity=0.6),
    hovertemplate='Real: %{x}<br>Predicho: %{y}'
))

# Puntos del mejor modelo Gradient Boosting 
fig_scatter.add_trace(go.Scatter(
    x=y_test, y=y_pred_gb_opt,
    mode='markers',
    name='GB Optimizado',
    marker=dict(color='green', size=8, opacity=0.8),
    hovertemplate='Real: %{x}<br>Predicho: %{y}'
))

# Puntos del Random Forest
fig_scatter.add_trace(go.Scatter(
    x=y_test, y=y_pred_rf_opt,
    mode='markers',
    name='RF Optimizado',
    marker=dict(color='yellow', size=8, opacity=0.8),
    hovertemplate='Real: %{x}<br>Predicho: %{y}'
))

# Puntos del SVR
fig_scatter.add_trace(go.Scatter(
    x=y_test, y=y_pred_svr_opt,
    mode='markers',
    name='SVR Optimizado',
    marker=dict(color='purple', size=8, opacity=0.8),
    hovertemplate='Real: %{x}<br>Predicho: %{y}'
))



fig_scatter.update_layout(
    title="Analisis de Residuales: Ventas Reales vs. Ventas Predichas",
    xaxis_title="Ventas Reales (Ground Truth)",
    yaxis_title="Ventas Predichas por el Modelo",
    hovermode='closest'
)
fig_scatter.show()

In [ ]:
# Grafico de dispersion en subplots: un eje por modelo, todos dentro de la misma figura
from plotly.subplots import make_subplots

modelos_a_graficar = [
    ('LR_Base', 'LR Base', 'red'),
    ('RF_Opt', 'RF Optimizado', 'goldenrod'),
    ('GB_Opt', 'GB Optimizado', 'green'),
    ('SVR_Opt', 'SVR Optimizado', 'purple'),
]

rango_min = min(y_test) - 2
rango_max = max(y_test) + 2

fig_scatter = make_subplots(
    rows=2, cols=2,
    subplot_titles=[nombre for _, nombre, _ in modelos_a_graficar]
)

posiciones = [(1, 1), (1, 2), (2, 1), (2, 2)]

for (clave, nombre, color), (fila, col) in zip(modelos_a_graficar, posiciones):
    # Linea ideal (prediccion perfecta, y = x)
    fig_scatter.add_trace(go.Scatter(
        x=[rango_min, rango_max], y=[rango_min, rango_max],
        mode='lines', line=dict(color='black', dash='dash'),
        showlegend=False
    ), row=fila, col=col)

    # Puntos de ese modelo
    fig_scatter.add_trace(go.Scatter(
        x=y_test, y=predicciones[clave],
        mode='markers',
        marker=dict(color=color, size=6, opacity=0.7),
        hovertemplate='Real: %{x}<br>Predicho: %{y}',
        showlegend=False
    ), row=fila, col=col)

fig_scatter.update_layout(
    title="Analisis de Residuales: Ventas Reales vs. Ventas Predichas por modelo",
    height=700,
    hovermode='closest'
)
fig_scatter.update_xaxes(title_text="Ventas Reales")
fig_scatter.update_yaxes(title_text="Ventas Predichas")
fig_scatter.show()


**Como leer los graficos:**
- **Grafico de barras:** fijate el salto entre la primera barra (Linear Regression base, sin intervencion humana) y el resto — ese salto es lo que aporta el tuneo de hiperparametros y probar otros algoritmos.
- **Por que Gradient Boosting y SVR:** Gradient Boosting es un ensamble secuencial que suele dominar en datos tabulares como este; SVR aporta una perspectiva geometrica distinta para la regresion.
- **Grafico de dispersion:** la linea punteada negra es un modelo perfecto (y = x). Los puntos verdes (modelo optimizado) deberian agruparse mas cerca de esa linea que los puntos rojos (modelo base), que suelen mostrar mas dispersion frente a valores atipicos.